In [ ]:
# ============================================================
# DISTRIBUTED ARITHMETIC - MULTIPLY-ACCUMULATE WITHOUT MULTIPLIERS
# ============================================================
#
# This notebook demonstrates the basic principle of distributed
# arithmetic for the computation
#
#       w = alpha1*x1 + alpha2*x2 + alpha3*x3
#
# without explicitly using multipliers.
#
# HOW TO USE THE NOTEBOOK
#
# 1. Use the coefficient sliders to select alpha1, alpha2, alpha3.
#
# 2. Use the input sliders to select x1, x2, x3.
#
# 3. Use the "Fraction bits K" slider to select the fixed-point
#    precision used for x1, x2, x3.
#
# 4. The notebook automatically constructs the ROM table
#
#       g(z1,z2,z3) = alpha1*z1 + alpha2*z2 + alpha3*z3
#
#    for all 2^3 = 8 possible binary addresses.
#
# 5. Use the "Clock cycle" slider to move through the bit columns
#    of x1, x2, x3.
#
# 6. At every clock cycle the notebook displays:
#
#       - the current bits x1j, x2j, x3j,
#       - the ROM address formed by these bits,
#       - the value gj read from the ROM,
#       - the shift operation,
#       - the updated accumulator.
#
# 7. After the last clock cycle, the distributed-arithmetic result
#    is compared with the direct computation
#
#       w = alpha1*x1 + alpha2*x2 + alpha3*x3.
#
# IMPORTANT
#
# This introductory version uses non-negative fixed-point inputs
# in the interval [0,1). This allows the fundamental mechanism of
# distributed arithmetic to be demonstrated without the additional
# sign-bit processing required by two's-complement representation.
#
# ============================================================


from ipywidgets import FloatSlider, IntSlider, HBox, VBox, Layout, HTML
from IPython.display import display


# ------------------------------------------------------------
# Global style sheet
# ------------------------------------------------------------

style_html = HTML("""
<style>

.da-root {
    font-family: monospace;
    width: 100%;
    max-width: 900px;
    box-sizing: border-box;
}

.da-title {
    font-size: 22px;
    font-weight: bold;
    margin-bottom: 8px;
    color: #1f1f1f;
}

.description-box {
    font-size: 13px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 9px;
    box-sizing: border-box;
    white-space: normal;
}

.section-box {
    border: 1px solid #c8d0dc;
    border-radius: 10px;
    padding: 10px 12px;
    background: #ffffff;
    box-sizing: border-box;
    width: 100%;
}

.section-title {
    font-size: 16px;
    font-weight: bold;
    margin-bottom: 7px;
    color: #243447;
}

.info {
    font-size: 14px;
    line-height: 1.6;
}

.metric {
    display: inline-block;
    min-width: 180px;
    font-weight: bold;
    color: #243447;
}

.bit-line {
    white-space: normal;
    line-height: 34px;
    margin-top: 5px;
    margin-bottom: 5px;
}

.bit-label {
    display: inline-block;
    width: 35px;
    font-weight: bold;
}

.bit-box {
    display: inline-block;
    width: 28px;
    height: 30px;
    line-height: 30px;
    text-align: center;
    margin-right: 3px;
    border-radius: 5px;
    border: 1px solid #7f8c9a;
    font-size: 13px;
    font-weight: bold;
    box-sizing: border-box;
}

.bit-normal {
    background: #dceeff;
    color: #0e3a66;
}

.bit-active {
    background: #fff3bf !important;
    color: #111111 !important;
    border: 3px solid #d62828 !important;
}

.address-box {
    display: inline-block;
    padding: 5px 10px;
    border: 2px solid #d62828;
    border-radius: 6px;
    background: #fff3bf;
    font-weight: bold;
    font-size: 16px;
}

.rom-value {
    display: inline-block;
    padding: 5px 10px;
    border: 1px solid #8f79a8;
    border-radius: 6px;
    background: #eee6f5;
    font-weight: bold;
    font-size: 16px;
}

.acc-value {
    display: inline-block;
    padding: 5px 10px;
    border: 1px solid #6b9b75;
    border-radius: 6px;
    background: #dff3e4;
    font-weight: bold;
    font-size: 16px;
}

.controls-title {
    font-family: monospace;
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.small-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    white-space: normal;
    margin-top: 5px;
}

.rom-table {
    border-collapse: collapse;
    font-family: monospace;
    font-size: 12px;
    width: 100%;
    line-height: 1.2;
}

.rom-table th,
.rom-table td {
    border: 1px solid #c8d0dc;
    padding: 4px 6px;
    text-align: center;
    vertical-align: middle;
}

.rom-table th {
    background: #f2f4f7;
}

.active-rom-row {
    background: #fff7d6;
    font-weight: bold;
}

.result-value {
    font-size: 18px;
    font-weight: bold;
    color: #1f3b4d;
}

</style>
""")


# ------------------------------------------------------------
# Fixed-point conversion
# ------------------------------------------------------------

def quantize_fraction(value, K):
    scale = 2**K
    q = int(round(value * scale))

    if q >= scale:
        q = scale - 1

    return q


def fraction_bits(value, K):
    q = quantize_fraction(value, K)
    return format(q, f'0{K}b')


def represented_fraction(value, K):
    q = quantize_fraction(value, K)
    return q / (2**K)


# ------------------------------------------------------------
# ROM table
# ------------------------------------------------------------

def build_rom(alpha1, alpha2, alpha3):
    table = {}

    for address in range(8):
        bits = format(address, '03b')

        z1 = int(bits[0])
        z2 = int(bits[1])
        z3 = int(bits[2])

        g = alpha1*z1 + alpha2*z2 + alpha3*z3

        table[bits] = g

    return table


# ------------------------------------------------------------
# Distributed arithmetic steps
# ------------------------------------------------------------

def distributed_arithmetic(alpha1, alpha2, alpha3, x1, x2, x3, K):
    b1 = fraction_bits(x1, K)
    b2 = fraction_bits(x2, K)
    b3 = fraction_bits(x3, K)

    rom = build_rom(alpha1, alpha2, alpha3)

    accumulator = 0.0
    steps = []

    for j in range(K):
        index = K - 1 - j

        bit1 = int(b1[index])
        bit2 = int(b2[index])
        bit3 = int(b3[index])

        address = f"{bit1}{bit2}{bit3}"

        g_value = rom[address]

        accumulator = 0.5 * (accumulator + g_value)

        steps.append({
            'cycle': j + 1,
            'bit_index': index,
            'x1_bit': bit1,
            'x2_bit': bit2,
            'x3_bit': bit3,
            'address': address,
            'g_value': g_value,
            'accumulator': accumulator
        })

    return b1, b2, b3, rom, steps


# ------------------------------------------------------------
# HTML helpers
# ------------------------------------------------------------

def render_bits(label, bit_string, active_index):
    boxes = []

    for i, bit in enumerate(bit_string):
        css = "bit-normal"

        if i == active_index:
            css += " bit-active"

        boxes.append(
            f"<span class='bit-box {css}'>{bit}</span>"
        )

    return f"""
    <div class="bit-line">
        <span class="bit-label">{label}</span>
        {''.join(boxes)}
    </div>
    """


def render_rom_table(rom, active_address):
    rows = []

    for address, value in rom.items():
        css = "active-rom-row" if address == active_address else ""

        rows.append(f"""
        <tr class="{css}">
            <td>{address}</td>
            <td>{address[0]}</td>
            <td>{address[1]}</td>
            <td>{address[2]}</td>
            <td>{value:.6f}</td>
        </tr>
        """)

    return ''.join(rows)


# ------------------------------------------------------------
# HTML containers
# ------------------------------------------------------------

title_html = HTML("""
<div class="da-root">
    <div class="da-title">
        Distributed Arithmetic
    </div>
</div>
""")


description_html = HTML("""
<div class="da-root">

    <div class="description-box">

        Distributed arithmetic computes a sum of products without
        explicitly using multiplier circuits. The coefficients are fixed,
        while combinations of coefficient sums are calculated in advance
        and stored in a ROM.<br>

        During each clock cycle, one bit from each input word forms the
        ROM address. The corresponding value of
        <b>g(z1,z2,z3)</b> is read and combined with the current accumulator
        through an add-and-right-shift operation.<br>

        Use the <b>Clock cycle</b> slider to follow the ROM lookup and
        accumulator update step by step.

    </div>

</div>
""")


input_html = HTML()
rom_html = HTML()
step_html = HTML()
result_html = HTML()


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(width='285px')

style_opts = {'description_width': '110px'}


alpha1_slider = FloatSlider(
    min=-2.0,
    max=2.0,
    step=0.1,
    value=0.8,
    description='Alpha 1:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


alpha2_slider = FloatSlider(
    min=-2.0,
    max=2.0,
    step=0.1,
    value=-0.5,
    description='Alpha 2:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


alpha3_slider = FloatSlider(
    min=-2.0,
    max=2.0,
    step=0.1,
    value=1.2,
    description='Alpha 3:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


x1_slider = FloatSlider(
    min=0.0,
    max=0.96875,
    step=0.03125,
    value=0.625,
    description='x1:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


x2_slider = FloatSlider(
    min=0.0,
    max=0.96875,
    step=0.03125,
    value=0.375,
    description='x2:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


x3_slider = FloatSlider(
    min=0.0,
    max=0.96875,
    step=0.03125,
    value=0.8125,
    description='x3:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


k_slider = IntSlider(
    min=3,
    max=8,
    step=1,
    value=5,
    description='Fraction bits K:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


clock_slider = IntSlider(
    min=1,
    max=k_slider.value,
    step=1,
    value=1,
    description='Clock cycle:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


controls_title = HTML("""
<div class="controls-title">
Controls
</div>
""")


controls_box = VBox(
    [
        controls_title,
        alpha1_slider,
        alpha2_slider,
        alpha3_slider,
        x1_slider,
        x2_slider,
        x3_slider,
        k_slider,
        clock_slider
    ],
    layout=Layout(
        width='315px',
        min_width='315px',
        border='1px solid #c8d0dc',
        padding='10px',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Main update function
# ------------------------------------------------------------

def update_display(*args):
    alpha1 = alpha1_slider.value
    alpha2 = alpha2_slider.value
    alpha3 = alpha3_slider.value

    K = k_slider.value

    x1 = represented_fraction(x1_slider.value, K)
    x2 = represented_fraction(x2_slider.value, K)
    x3 = represented_fraction(x3_slider.value, K)

    clock = clock_slider.value

    b1, b2, b3, rom, steps = distributed_arithmetic(
        alpha1,
        alpha2,
        alpha3,
        x1,
        x2,
        x3,
        K
    )

    current = steps[clock - 1]

    active_index = current['bit_index']

    direct_result = alpha1*x1 + alpha2*x2 + alpha3*x3

    da_result = steps[-1]['accumulator']


    # --------------------------------------------------------
    # Input section
    # --------------------------------------------------------

    input_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            Coefficients and Fixed-Point Inputs
        </div>

        <div class="info">

            <span class="metric">Coefficients</span>
            α1 = {alpha1:.3f},
            α2 = {alpha2:.3f},
            α3 = {alpha3:.3f}
            <br>

            <span class="metric">Fraction bits</span>
            K = {K}
            <br>

            <span class="metric">Represented inputs</span>
            x1 = {x1:.6f},
            x2 = {x2:.6f},
            x3 = {x3:.6f}

        </div>

        <div style="margin-top:7px;">

            {render_bits('x1', b1, active_index)}

            {render_bits('x2', b2, active_index)}

            {render_bits('x3', b3, active_index)}

        </div>

        <div class="small-note">
            The highlighted column contains the three input bits
            used to form the ROM address during the current cycle.
        </div>

    </div>
    """


    # --------------------------------------------------------
    # ROM table
    # --------------------------------------------------------

    rom_rows = render_rom_table(
        rom,
        current['address']
    )

    rom_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            ROM Table
        </div>

        <div class="small-note">
            g(z1,z2,z3) = α1 z1 + α2 z2 + α3 z3
        </div>

        <table class="rom-table">

            <tr>
                <th>Address</th>
                <th>z1</th>
                <th>z2</th>
                <th>z3</th>
                <th>g(z1,z2,z3)</th>
            </tr>

            {rom_rows}

        </table>

    </div>
    """


    # --------------------------------------------------------
    # Current clock cycle
    # --------------------------------------------------------

    previous_accumulator = 0.0

    if clock > 1:
        previous_accumulator = steps[clock - 2]['accumulator']

    step_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            Current Clock Cycle
        </div>

        <div class="info">

            <span class="metric">Clock cycle</span>
            {clock} of {K}
            <br>

            <span class="metric">Selected bits</span>
            {current['x1_bit']},
            {current['x2_bit']},
            {current['x3_bit']}
            <br>

            <span class="metric">ROM address</span>
            <span class="address-box">
                {current['address']}
            </span>

        </div>

        <div class="info" style="margin-top:9px;">

            <span class="metric">ROM output g</span>
            <span class="rom-value">
                {current['g_value']:.6f}
            </span>

            <br><br>

            <span class="metric">Previous accumulator</span>
            {previous_accumulator:.6f}

            <br>

            <span class="metric">Add</span>
            {previous_accumulator:.6f}
            +
            {current['g_value']:.6f}
            =
            {previous_accumulator + current['g_value']:.6f}

            <br>

            <span class="metric">Right shift / divide by 2</span>
            ({previous_accumulator + current['g_value']:.6f}) / 2

            <br>

            <span class="metric">New accumulator</span>
            <span class="acc-value">
                {current['accumulator']:.6f}
            </span>

        </div>

    </div>
    """


    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------

    if clock < K:

        status_text = f"""
        <div class="small-note">
            The distributed-arithmetic computation is still in progress.
            {K - clock} bit column(s) remain.
        </div>
        """

        final_text = ""

    else:

        error = da_result - direct_result

        status_text = """
        <div class="small-note">
            <b>The distributed-arithmetic computation is complete.</b>
        </div>
        """

        final_text = f"""
        <div class="info" style="margin-top:8px;">

            <span class="metric">Distributed arithmetic</span>
            <span class="result-value">
                {da_result:.8f}
            </span>

            <br>

            <span class="metric">Direct computation</span>
            <span class="result-value">
                {direct_result:.8f}
            </span>

            <br>

            <span class="metric">Difference</span>
            {error:.3e}

        </div>
        """


    result_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            Multiply-Accumulate Result
        </div>

        <div class="info">

            Direct expression:
            <br>

            <b>
            w = α1 x1 + α2 x2 + α3 x3
            </b>

            <br>

            w =
            ({alpha1:.3f})({x1:.6f})
            +
            ({alpha2:.3f})({x2:.6f})
            +
            ({alpha3:.3f})({x3:.6f})

        </div>

        {status_text}

        {final_text}

    </div>
    """


# ------------------------------------------------------------
# Reset clock when parameters change
# ------------------------------------------------------------

def reset_clock_and_update(change):
    if clock_slider.value != 1:
        clock_slider.value = 1
    else:
        update_display()


def update_fraction_bits(change):
    K = change['new']

    clock_slider.max = K

    step = 2**(-K)

    maximum = 1.0 - step

    x1_slider.step = step
    x2_slider.step = step
    x3_slider.step = step

    x1_slider.max = maximum
    x2_slider.max = maximum
    x3_slider.max = maximum

    if x1_slider.value > maximum:
        x1_slider.value = maximum

    if x2_slider.value > maximum:
        x2_slider.value = maximum

    if x3_slider.value > maximum:
        x3_slider.value = maximum

    if clock_slider.value != 1:
        clock_slider.value = 1
    else:
        update_display()


alpha1_slider.observe(reset_clock_and_update, names='value')
alpha2_slider.observe(reset_clock_and_update, names='value')
alpha3_slider.observe(reset_clock_and_update, names='value')

x1_slider.observe(reset_clock_and_update, names='value')
x2_slider.observe(reset_clock_and_update, names='value')
x3_slider.observe(reset_clock_and_update, names='value')

k_slider.observe(update_fraction_bits, names='value')

clock_slider.observe(update_display, names='value')


# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

input_html.layout = Layout(
    width='565px',
    min_width='565px',
    overflow='visible'
)


top_row = HBox(
    [
        input_html,
        controls_box
    ],
    layout=Layout(
        width='900px',
        max_width='900px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='visible'
    )
)


rom_html.layout = Layout(
    width='430px',
    min_width='430px',
    overflow='visible'
)


step_html.layout = Layout(
    width='450px',
    min_width='450px',
    overflow='visible'
)


middle_row = HBox(
    [
        rom_html,
        step_html
    ],
    layout=Layout(
        width='900px',
        max_width='900px',
        align_items='flex-start',
        justify_content='space-between',
        overflow='visible'
    )
)


result_html.layout = Layout(
    width='900px',
    max_width='900px',
    overflow='visible'
)


main_layout = VBox(
    [
        top_row,
        middle_row,
        result_html
    ],
    layout=Layout(
        width='900px',
        max_width='900px',
        align_items='flex-start',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Initial display
# ------------------------------------------------------------

update_display()

display(style_html)
display(title_html)
display(description_html)
display(main_layout)